### Classes Game

Vous trouverez ici les dernières versions des classes de base du jeu.

In [ ]:
class Account :
    def __init__(self, cash=100.0):
        '''
        @param cash : cash initial de l'utilisateur
        @param isPart : possible d'acheter/vendre des fractions de la valeur mobilière
        
        '''
        self.cash = cash

        self.cashInvest = self.cash 
        self.security = 0
        self.tradingFees = 0.0

    def computeTradingFees(self, numberOfSecurity, unityPrice):

        totalPrice = numberOfSecurity * unityPrice

        if(totalPrice > 500):
            tradingFees = 1.99
        else:
            tradingFees = 0.006 * totalPrice

        return tradingFees

    """
    def buy(self, numberOfSecurity, unityPrice):

        if unityPrice < 0:
            raise NameError("Account - buy : unityPrice can not be negative")

        tradingFees = self.computeTradingFees(numberOfSecurity, unityPrice)
        cost = unityPrice * numberOfSecurity + tradingFees
        
        if(self.cash - cost >= 0):
            self.cash = self.cash - cost
            self.security = self.security + numberOfSecurity
            self.tradingFees = self.tradingFees + tradingFees
        else:
            raise NameError("Account - buy : not enought money")

    def buyMax(self, unityPrice):

        numberOfSecurity = int(self.cash / unityPrice)

        self.buy(numberOfSecurity, unityPrice)

    def sell(self, numberOfSecurity, unityPrice):

        if(self.security - numberOfSecurity >= 0):
            tradingFees = self.computeTradingFees(numberOfSecurity, unityPrice)
            self.security = self.security - numberOfSecurity
            self.cash = self.cash + numberOfSecurity * unityPrice - tradingFees
            self.tradingFees = self.tradingFees + tradingFees
        else:
            raise NameError("Sell error : not enought coins: ", self.security, " but ask to sell ", numberOfSecurity )

    def sellMax(self, unityPrice):
        self.sell(self.security, unityPrice)
    """
    def invest(self, cash, unityPrice):
        if unityPrice < 0:
            raise NameError("Account - invest : unityPrice can not be negative")

        numberOfSecurity = cash // unityPrice
        tradingFees = self.computeTradingFees(numberOfSecurity, unityPrice)
        cost = unityPrice * numberOfSecurity + tradingFees
        
        if(self.cash - cost >= 0):
            self.cash = self.cash - cash
            self.security = self.security + cash / unityPrice
            self.tradingFees = self.tradingFees + tradingFees
        else:
            raise NameError("Account - invest : not enought money")

    def investAll(self, unityPrice):

        numberOfSecurity = self.cash // unityPrice  
        tradingFees = self.computeTradingFees(numberOfSecurity, unityPrice)
        cashToInvest = self.cash - tradingFees

        self.invest(cashToInvest, unityPrice)
    
    def desinvest(self, numberOfSecurity, unityPrice):
        if unityPrice < 0:
            raise NameError("Account - desinvest : unityPrice can not be negative")

        tradingFees = self.computeTradingFees(numberOfSecurity, unityPrice)
        
        if(self.security - numberOfSecurity >= 0 and self.cash > tradingFees):
            self.security = self.security - numberOfSecurity
            self.cash = self.cash + numberOfSecurity * unityPrice - tradingFees
            self.tradingFees = self.tradingFees + tradingFees
        else:
            raise NameError("Account - desinvest : not enought securities")
        
    def desinvestAll(self, unityPrice):
        self.desinvest(self.security, unityPrice)

    def cashIn(self, cash):
        if cash < 0:
            raise NameError("Account - cahsIn : cash can not be negative")
        self.cashInvest = self.cashInvest + cash
        self.cash = self.cash + cash

    def getTheoricalValue(self, unityPrice):
        return (self.cash + self.security * unityPrice)
    
    def getTheoricalGain(self, unityPrice):
        return self.getTheoricalValue(unityPrice) - self.cashInvest - self.tradingFees

    ''' 
    Méthode permettant d'afficher vos objets Account de manière conviviale. Cette méthode est
    automatiquement appelée quand un objet Account est donné en argument de la fonction 'print'
    '''
    def __str__(self):
        return "Cash courant: "+str(self.cash)+" & Security: "+str(self.security)+" & Cash investi: "+str(self.cashInvest)

### Game simple

Classe pour un jeu réalisant une prédiction seulement en fonction du prix journalier.

La série en paramètre est le prix journalier

Exemple d'utilisation:

    # création Account et Strategy
    account = Account(100)
    strategy = StrategyOracle()

    # création du jeu
    game = Game(serie.Close)
    game.setAccount(account)
    game.setStrategy(strategy)
    game.run()

In [ ]:

class Game :

    '''
    Définition des variables appartenant à la classe, et non pas aux objets
    '''
    BUY = 0
    SELL = 1
    STILL = 2
    INVEST = 3
    DESINVEST = 4
    INVEST_ALL = 5
    DESINVEST_ALL = 6
    CASH_IN = 7

    ACTIONS = ["BUY", "SELL", "STILL", "INVEST", "DESINVEST", "INVEST_ALL", "DESINVEST_ALL", "CASH_IN"]

    @staticmethod
    def printAction(action):
        return Game.ACTIONS[action]

    def __init__(self, prices, account=None, strategy=None, data=None):
        '''
        @param prices: Pandas series with the prices, indexed by date
        @param account: Account object 
        @param strategy: Strategy object to buy/sell/still everyday
        @param data: Pandas dataframe with the data to use to make the prediction. If None, prices is used.
        '''
        self.dates = prices.index

        self.prices = prices.to_numpy()
        
        self.data = data
        if self.data is None:
            self.data = self.prices
        else:
            self.data = data.to_numpy()
           
        self.account = account
        self.strategy = strategy
        self.clock = 0
        
    def setStrategy(self, strategy):
        self.strategy = strategy

    def setAccount(self, account):
        self.account = account
        self.clock = 0

    def run(self, debug=False):

        if(self.account is None):
            raise NameError("Account is not set")
        
        if(self.strategy is None):
            raise NameError("Strategy is not set")
        
        for i in range(0, len(self.data)):
            
            unityPrice = self.prices[i]
            unityData = self.data[i]

            actions = self.strategy.getAction(self, unityPrice, self.account, self.dates[self.clock], unityData=unityData, debug=debug)

            if( type(actions) is tuple):
                actions = [actions]

            for action in actions:
                self.processAction(action, unityPrice, debug)

            if debug:
                print(self.account)
                
            self.clock = self.clock+1
    
    def processAction(self, actionToProcess, unityPrice, debug):
        
        (action, amount) = actionToProcess

        actionStr = Game.ACTIONS[action]

        if debug:
            print("--- ", actionStr, " ",amount, "securities at clock ", self.clock)

        try:
            if(action == Game.BUY):
                self.account.buy(amount, unityPrice)
            elif(action == Game.SELL):
                self.account.sell(amount, unityPrice)
            elif(action == Game.INVEST):
                self.account.invest(amount, unityPrice)
            elif(action == Game.DESINVEST):
                self.account.desinvest(amount, unityPrice)
            elif(action == Game.INVEST_ALL):
                self.account.investAll(unityPrice)
            elif(action == Game.DESINVEST_ALL):
                self.account.desinvestAll(unityPrice)
            elif(action == Game.CASH_IN):
                self.account.cashIn(amount)
            else:
                pass
        except NameError as error:
                print(error)

    def getTheoricalValue(self):
        return self.account.getTheoricalValue(self.prices[self.clock-1])
    
    def getTheoricalGain(self):
        return self.account.getTheoricalGain(self.prices[self.clock-1])

    def getPandasSerie(self):
        return pd.Series(self.prices, index=self.dates)

    def __str__(self):
        return "Game after "+str(self.clock)+"/"+str(len(self.prices))+" steps - "+str(self.account)
    

### Classe IStrategy

In [1]:
class IStrategy :

    def __init__(self, name="IStrategy", _isRandom=False):
        self.name = name
        self._isRandom = _isRandom

    def isRandom(self):
        return self._isRandom

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        '''
        @return (action, amount)
        '''
        pass

    def getName(self):
        '''
        @return the name of the strategy
        '''
        return self.name